# 03 · Gold — Indicadores analíticos Spark SQL

**Proyecto:** PoC Arquitectura Lakehouse para Monitoreo Ambiental Hídrico  
**Autora:** Kathiana Murillo García  
**Máster:** UNIR — Análisis y Visualización de Datos Masivos (2026–2027)  

## Objetivo
Construir dos indicadores analíticos Gold a partir de los registros VALID de la capa Silver:

- **Indicador 1 (PI2):** Índice de Oxigenación Mensual por Estación  
- **Indicador 2 (PI3):** Tendencia de Eutrofización Anual (T-P, Chl-a)

Ambos indicadores se construyen con **Spark SQL** sobre la vista temporal
`silver_kasumigaura`, creada a partir de los registros VALID de la tabla Silver.

_Revisado: umbrales OECD confirmados para lago templado._

## 1. Configuración

In [ ]:
# Celda 1 — Parámetros
from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import pandas as pd

CATALOG = "training"
SCHEMA  = "kasumigaura"
TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"
TABLE_GOLD_OX = f"{CATALOG}.{SCHEMA}.gold_oxigenacion_mensual"
TABLE_GOLD_EU = f"{CATALOG}.{SCHEMA}.gold_eutrofizacion_anual"

try:
    CATALOG = spark.sql("SELECT current_catalog()").collect()[0][0]
    TABLE_SILVER = f"{CATALOG}.{SCHEMA}.silver_water_quality"
    TABLE_GOLD_OX = f"{CATALOG}.{SCHEMA}.gold_oxigenacion_mensual"
    TABLE_GOLD_EU = f"{CATALOG}.{SCHEMA}.gold_eutrofizacion_anual"
except Exception:
    pass

print(f"Silver : {TABLE_SILVER}")
print(f"Gold Ox: {TABLE_GOLD_OX}")
print(f"Gold Eu: {TABLE_GOLD_EU}")

## 2. Vista temporal de registros VALID

In [ ]:
# Celda 2 — Crear vista temporal `silver_kasumigaura` con solo registros VALID
spark.table(TABLE_SILVER) \
    .filter(F.col("dq_flag") == "VALID") \
    .createOrReplaceTempView("silver_kasumigaura")

n_valid = spark.sql("SELECT COUNT(*) FROM silver_kasumigaura").collect()[0][0]
print(f"Vista silver_kasumigaura: {n_valid:,} registros VALID")

## 3. Indicador 1 — Índice de Oxigenación Mensual por Estación

In [ ]:
# Celda 3 — Construir Indicador 1 con Spark SQL
# Clasifica el estado ecológico según umbrales limnológicos (Wetzel, 2001):
#   hipoxia_critica : DO0m < 4 mg/L
#   aceptable       : 4 ≤ DO0m < 6 mg/L
#   optimo          : DO0m ≥ 6 mg/L

spark.sql(f"""
CREATE OR REPLACE TABLE {TABLE_GOLD_OX} AS
SELECT
    `ST.`                               AS station,
    YEAR(Date)                          AS year,
    MONTH(Date)                         AS month,
    ROUND(AVG(DO0m), 2)                 AS avg_do_mg_l,
    ROUND(MIN(DO0m), 2)                 AS min_do_mg_l,
    ROUND(MAX(DO0m), 2)                 AS max_do_mg_l,
    ROUND(STDDEV(DO0m), 3)              AS std_do_mg_l,
    COUNT(*)                            AS n_obs,
    CASE
        WHEN AVG(DO0m) < 4  THEN 'hipoxia_critica'
        WHEN AVG(DO0m) < 6  THEN 'aceptable'
        ELSE 'optimo'
    END                                 AS estado_oxigenacion
FROM silver_kasumigaura
WHERE DO0m IS NOT NULL
GROUP BY `ST.`, YEAR(Date), MONTH(Date)
ORDER BY station, year, month
""")

n_ox = spark.table(TABLE_GOLD_OX).count()
print(f"✓ {TABLE_GOLD_OX}: {n_ox:,} registros")
spark.table(TABLE_GOLD_OX).show(5)